## Carga de datos en base de datos MySQL

### Objetivo
Este notebook carga los datasets procesados en una base de datos MySQL relacional, estableciendo la estructura de **modelo estrella** necesaria para el análisis de resiliencia empresarial.

### Estructura de la base de datos
- **Base de datos**: `ipc_analisis_empresarial`
- **Tablas de dimensiones**: `territorio`, `tiempo`, `tipo_medida`, `sectores_ipc`
- **Tablas de hechos**: `ipc`, `empresas_constituidas`, `empresas_disueltas`

### Metodología
1. **Conexión** a MySQL mediante `sqlalchemy` usando variables de entorno (`.env`).
2. **Creación** de la base de datos si no existe.
3. **Carga secuencial** de tablas dimensión y asignación de claves primarias.
4. **Carga de tablas de hechos** con asignación de claves primarias (autoincremental para `ipc`).
5. **Establecimiento de relaciones** — Creación de claves foráneas que vinculan las tablas de hechos con sus dimensiones correspondientes.

In [ ]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text

# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación de módulos de carga de datos
import src.load.load_db as db

# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)


# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())
from dotenv import load_dotenv

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: H:\Cursos\Adalab_Analista & IA\taller_git\spain-business-resilience-analytics\notebooks


True

Cargamos los dataframes

In [2]:
df_empr_const = pd.read_csv('../files/data_processed/empresas_constituidas.csv')
df_empr_dis = pd.read_csv('../files/data_processed/empresas_disueltas.csv')
df_ipc = pd.read_csv('../files/data_processed/ipc.csv')
df_sectores_ipc = pd.read_csv('../files/data_processed/sectores_ipc.csv')
df_territorio = pd.read_csv('../files/data_processed/territorio.csv')
df_tiempo = pd.read_csv('../files/data_processed/tiempo.csv')
df_tipo_medida = pd.read_csv('../files/data_processed/tipo_medida.csv')

- Limpieza inicial

- Elimina las tablas existentes en orden de dependencia inversa para evitar conflictos con claves foráneas. Permite una recarga completa desde cero.

In [3]:
db.drop_all_tables("ipc_analisis_empresarial", [
    "empresas_disueltas",
    "empresas_constituidas",
    "ipc",
    "tipo_medida",
    "sectores_ipc",
    "tiempo",
    "territorio",
])

Cargamos los df en la bbdd

In [4]:
conn_str = db.get_connection_string(db_name=None)

print(f"✅ Conexión configurada correctamente: {conn_str.split(':')[0]}@****")
 

✅ Conexión configurada correctamente: mysql+pymysql@****


In [5]:
db.create_database_if_not_exists("ipc_analisis_empresarial")

In [6]:
db.load_dataframe_to_mysql(df_territorio, "territorio", "ipc_analisis_empresarial")

In [7]:
db.set_primary_key("territorio", "id_territorio", "ipc_analisis_empresarial")

In [8]:
db.load_dataframe_to_mysql(df_tiempo, "tiempo", "ipc_analisis_empresarial")

In [9]:
db.set_primary_key("tiempo", "id_tiempo", "ipc_analisis_empresarial")

In [10]:
db.load_dataframe_to_mysql(df_tipo_medida, "tipo_medida", "ipc_analisis_empresarial")

In [11]:
db.set_primary_key("tipo_medida", "id_medida", "ipc_analisis_empresarial")

In [12]:
db.load_dataframe_to_mysql(df_sectores_ipc, "sectores_ipc", "ipc_analisis_empresarial")

In [13]:
db.set_primary_key("sectores_ipc", "id_sector", "ipc_analisis_empresarial")

In [14]:
db.load_dataframe_to_mysql(df_ipc, "ipc", "ipc_analisis_empresarial")

In [15]:
db.add_autoincrement_id("ipc", "ipc_analisis_empresarial")

In [16]:
db.set_foreign_keys(
    fact_table="ipc",
    relations=[
        {"fk_column": "id_territorio", "dimension_table": "territorio"},
        {"fk_column": "id_tiempo", "dimension_table": "tiempo"},
        {"fk_column": "id_medida", "dimension_table": "tipo_medida"},
        {"fk_column": "id_sector", "dimension_table": "sectores_ipc"},
    ],
    db_name="ipc_analisis_empresarial"
)

In [17]:
db.load_dataframe_to_mysql(df_empr_const, "empresas_constituidas", "ipc_analisis_empresarial")

In [18]:
db.set_primary_key("empresas_constituidas", "id_const", "ipc_analisis_empresarial")

In [19]:
db.set_foreign_keys(
    fact_table="empresas_constituidas",
    relations=[
        {"fk_column": "id_territorio", "dimension_table": "territorio"},
        {"fk_column": "id_tiempo", "dimension_table": "tiempo"},

    ],
    db_name="ipc_analisis_empresarial"
)

In [20]:
db.load_dataframe_to_mysql(df_empr_dis, "empresas_disueltas", "ipc_analisis_empresarial")

In [21]:
db.set_primary_key("empresas_disueltas", "id_dis", "ipc_analisis_empresarial")

In [22]:
db.set_foreign_keys(
    fact_table="empresas_disueltas",
    relations=[
        {"fk_column": "id_territorio", "dimension_table": "territorio"},
        {"fk_column": "id_tiempo", "dimension_table": "tiempo"},

    ],
    db_name="ipc_analisis_empresarial"
)